# Generate a Sonnet from a Starting Phrase

This notebook trains a small LSTM on a handful of Shakespeare sonnets, building training sequences from each whole sonnet (not just single lines). It then writes a longer, sonnet-length passage word by word from a starting phrase, using temperature sampling so the output is less repetitive.

**How to run:** Open in Google Colab, choose **Runtime > Change runtime type > GPU**, then **Runtime > Run all**.

_Rewritten from my original Colab notebook: `sek.Sonnet predict.ipynb`._

## 1. Imports

In [ ]:
import re
import urllib.request

import numpy as np
import tensorflow as tf
import keras
from keras import layers

SEED = 42
keras.utils.set_random_seed(SEED)  # seeds Python, NumPy and TensorFlow
print("TensorFlow:", tf.__version__)

## 2. Load sonnets and split them into individual poems
Sonnets in the Gutenberg file are separated by Roman-numeral headings.

In [ ]:
def download_gutenberg_book(url):
    """Download a plain-text book from Project Gutenberg and remove the licence header/footer."""
    raw = urllib.request.urlopen(url).read().decode("utf-8-sig")
    start = raw.find("*** START OF")
    end = raw.find("*** END OF")
    if start != -1 and end != -1:
        raw = raw[raw.find("\n", start) + 1:end]
    return raw.replace("\r\n", "\n").strip()

In [ ]:
book = download_gutenberg_book("https://www.gutenberg.org/cache/epub/1041/pg1041.txt")
parts = re.split(r"\n\s*[IVXLC]+\s*\n", "\n" + book + "\n")
sonnets = [p.strip() for p in parts if len(p.split("\n")) >= 12]
sonnets = sonnets[:30]  # keep it small and fast for a beginner project
print("Sonnets used:", len(sonnets))
print(sonnets[0])

## 3. Vocabulary and training windows
Each sonnet becomes one long word list. We slide a fixed window of `WINDOW` words over it and predict the following word.

In [ ]:
def clean_words(line):
    """Lower-case a line and keep only words (letters and apostrophes)."""
    line = line.lower().replace("\u2019", "'")
    return re.findall(r"[a-z']+", line)


def build_vocab(lines):
    """Map every word to an integer id. Id 0 is reserved for padding."""
    words = sorted({w for line in lines for w in clean_words(line)})
    word_to_id = {w: i + 1 for i, w in enumerate(words)}
    id_to_word = {i: w for w, i in word_to_id.items()}
    return word_to_id, id_to_word


def encode(line, word_to_id):
    return [word_to_id[w] for w in clean_words(line) if w in word_to_id]

In [ ]:
WINDOW = 10
word_to_id, id_to_word = build_vocab(sonnets)
vocab_size = len(word_to_id) + 1

X_list, y_list = [], []
for sonnet in sonnets:
    ids = encode(sonnet, word_to_id)
    for i in range(1, len(ids)):
        X_list.append(ids[max(0, i - WINDOW):i])
        y_list.append(ids[i])

X = keras.utils.pad_sequences(X_list, maxlen=WINDOW, padding="pre")
y = np.array(y_list)
print("X:", X.shape, "| y:", y.shape, "| vocabulary:", vocab_size)

## 4. Model and training

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(WINDOW,)),
    layers.Embedding(vocab_size, 128),
    layers.LSTM(256),
    layers.Dropout(0.2),
    layers.Dense(vocab_size, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ModelCheckpoint("best_sonnet_model.keras", monitor="val_loss", save_best_only=True),
]
history = model.fit(X, y, epochs=60, batch_size=64, validation_split=0.1, callbacks=callbacks)

## 5. Evaluate

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history["loss"], label="train loss")
if "val_loss" in history.history:
    plt.plot(history.history["val_loss"], label="validation loss")
plt.xlabel("epoch")
plt.legend()
plt.show()

## 6. Generate a sonnet
`temperature` < 1 makes the text safer, > 1 makes it more surprising.

In [ ]:
def sample(probs, temperature=0.8):
    probs = np.asarray(probs, dtype="float64")
    probs[0] = 0  # skip padding
    logits = np.log(probs + 1e-9) / temperature
    probs = np.exp(logits) / np.sum(np.exp(logits))
    return int(np.random.choice(len(probs), p=probs))


def generate_sonnet(start_phrase, n_words=100, words_per_line=8, temperature=0.8):
    ids = encode(start_phrase, word_to_id)
    words = clean_words(start_phrase)
    for _ in range(n_words):
        x = keras.utils.pad_sequences([ids[-WINDOW:]], maxlen=WINDOW, padding="pre")
        next_id = sample(model.predict(x, verbose=0)[0], temperature)
        ids.append(next_id)
        words.append(id_to_word[next_id])
    lines = [" ".join(words[i:i + words_per_line]) for i in range(0, len(words), words_per_line)]
    return "\n".join(lines)


print(generate_sonnet("shall i compare thee to a summer's day"))